In [7]:
import pandas as pd
from pathlib import Path

df_w = pd.read_csv("../../1-transaction-data/price_series.csv", index_col="date", parse_dates=True).asfreq("W-SUN")

folder = Path("raw_weekly_indexes")
out_name = "../weekly_sar_aligned.csv"

canonical = {
    "business_bay": "Business Bay",
    "damac_hills_2": "DAMAC HILLS 2",
    "damac_hills": "DAMAC HILLS",
    "dow_square": "TOWN SQUARE",
    "downtown_dubai": "DownTown Dubai",
    "dubai_creek_harbour": "Dubai Creek Harbour",
    "dubai_marina": "Dubai Marina",
    "dubai_south_residential_district": "Dubai South Residential District",
    "dubai_sports_city": "Dubai Sports City",
    "geometry": "Al Furjan",
    "international_city_phase_1": "International City Phase 1",
    "jumeirah_lake_towers": "Jumeirah Lakes Towers",
    "jumeirah_village_circle": "Jumeirah Village Circle",
    "jumeirah_village_triangle": "Jumeirah Village Triangle",
    "jumeriah_beach_residence": "Jumeriah Beach Residence  - JBR",
    "mudon": "Mudon",
    "palm_jumeirah": "Palm Jumeirah",
    "silicon_oasis": "Silicon Oasis",
    "the_greens": "The Greens"
}

required_cols = {"week_start", "vv_mean_db", "vh_mean_db", "vv_vh_ratio_mean_db"}
base_features = ["vv_mean_db", "vh_mean_db", "vv_vh_ratio_mean_db"]
lags = [4, 8, 12, 20]

def load_sat_csv(stem):
    csv_path = folder / f"{stem}_weekly_s1sar.csv"
    if not csv_path.exists():
        return None
    df_sat = pd.read_csv(csv_path, parse_dates=["week_start"])
    if not required_cols.issubset(df_sat.columns):
        return None
    df_sat = df_sat.copy()
    df_sat["week_start"] = pd.to_datetime(df_sat["week_start"])
    return df_sat

def build_sar_weekly_series(df_sat, feat):
    s = (
        df_sat
        .set_index("week_start")[feat]
        .astype(float)
        .sort_index()
        .resample("W-SUN").mean()
        .rolling(4, min_periods=1).mean()
    )
    return s

def build_full_weekly_index(series_list):
    mins = [s.index.min() for s in series_list if s is not None and len(s.index) > 0]
    maxs = [s.index.max() for s in series_list if s is not None and len(s.index) > 0]
    if not mins or not maxs:
        return pd.DatetimeIndex([])
    start = min(mins)
    end = max(maxs)
    return pd.date_range(start=start, end=end, freq="W-SUN")

def add_lags(df_base, lags_list):
    lag_frames = []
    for L in lags_list:
        shifted = df_base.shift(L)
        shifted.columns = [f"{c}_lag{L}" for c in shifted.columns]
        lag_frames.append(shifted)
    if not lag_frames:
        return df_base.copy()
    return pd.concat([df_base] + lag_frames, axis=1)

series_for_index = []
region_series = {}

for stem, region in canonical.items():
    df_sat = load_sat_csv(stem)
    if df_sat is None:
        continue
    for feat in base_features:
        s = build_sar_weekly_series(df_sat, feat)
        region_series[(region, feat)] = s
        series_for_index.append(s)

sar_index = build_full_weekly_index(series_for_index)
if len(sar_index) == 0:
    raise RuntimeError("No SAR series could be loaded")

columns_dict = {}
for (region, feat), s in region_series.items():
    s_aligned = s.reindex(sar_index)
    s_filled = s_aligned.ffill()
    columns_dict[f"{region}__{feat}"] = s_filled

df_sar_base = pd.DataFrame(columns_dict).reindex(sar_index)

for feat in base_features:
    feat_cols = [c for c in df_sar_base.columns if c.endswith(f"__{feat}")]
    if feat_cols:
        df_sar_base[f"Global__{feat}"] = df_sar_base[feat_cols].mean(axis=1)

df_sar_all = add_lags(df_sar_base, lags)
df_sar_all = df_sar_all.apply(pd.to_numeric, errors="coerce")

df_aligned = df_sar_all.reindex(df_w.index)

nans_total = int(df_aligned.isna().sum().sum())
if nans_total != 0:
    raise RuntimeError(f"Unexpected NaNs after alignment: {nans_total}")

df_aligned.to_csv(out_name)

print("Final SAR feature export completed")
print(f"- Output file: {out_name}")
print(f"- Date range: {df_aligned.index.min().date()} to {df_aligned.index.max().date()}")
print(f"- Rows: {len(df_aligned):,} | Columns: {df_aligned.shape[1]:,}")
print(f"- Lags used: {lags}")


Final SAR feature export completed
- Output file: ../weekly_sar_aligned.csv
- Date range: 2015-09-06 to 2025-09-28
- Rows: 526 | Columns: 300
- Lags used: [4, 8, 12, 20]
